In [1]:
import pathlib
import h5py
import numpy as np

from text_embeddings.clustering import bootstrap_clustering_metrics

In [2]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

In [3]:
filename = 'data/embeddings/modernbert-distilbert-1.5b_embeddings.h5'

In [4]:
filepath = pathlib.Path(filename)
ds_name = pathlib.PurePath(filepath.stem).name
output_dir = pathlib.Path('data') / 'plots' / ds_name
output_dir

PosixPath('data/plots/modernbert-distilbert-1.5b_embeddings')

In [5]:
output_dir.mkdir(parents=True)

In [6]:
!ls -lh {filename}

ls: cannot access 'data/embeddings/modernbert-distilbert-1.5b_embeddings.h5': No such file or directory


In [7]:
fin = h5py.File(filename, 'r')

FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = 'data/embeddings/modernbert-distilbert-1.5b_embeddings.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

In [ ]:
# Read embeddings to numpy array

def read_to_numpy(fin: h5py.File, name: str):
    ds = fin[name]
    array = np.empty(ds.shape, dtype=ds.dtype)
    ds.read_direct(array)
    return array


embeddings = read_to_numpy(fin, 'embeddings')
labels = read_to_numpy(fin, 'labels')

In [ ]:
texts = np.array(fin['texts'].asstr())

# T-SNE projection with labels

In [ ]:
from sklearn.manifold import TSNE

In [ ]:
%%time

tsne = TSNE(
    n_components=2,
    learning_rate=500,
    init='random',
    perplexity=100,
    metric='cosine',
)
projected = tsne.fit_transform(embeddings)

In [ ]:
fig, ax = plt.subplots(figsize=(12,10))

sns.scatterplot(x=projected[:, 0], y=projected[:, 1], hue=labels, palette='tab20b')
fig.savefig(str(output_dir/ 'tsne.png'))

# K-Means

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
from tqdm import tqdm, trange

In [ ]:
max_k = 30

In [ ]:
ks = list(range(2, max_k + 1))

In [ ]:
models = {}
silhouette_scores = []
rand_scores = []

for k in tqdm(ks):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(embeddings)
    sil_score = silhouette_score(embeddings, cluster_labels)
    silhouette_scores.append(sil_score)

    rand = adjusted_rand_score(labels, cluster_labels)
    rand_scores.append(rand)
    
    models[k] = kmeans

In [ ]:
import pandas as pd

In [ ]:
pd.DataFrame({'k': ks, 'silhouette': silhouette_scores, 'rand': rand_scores})

In [ ]:
%%time
result = bootstrap_clustering_metrics(embeddings, labels)

In [ ]:
report = f"""
Silhouette
==========

mean: {float(result["silhouette"]["mean"]):.4f}
std:  {float(result["silhouette"]["std"]):.4f}


Adjusted Rand Index
===================

mean: {float(result["adjusted_rand_index"]["mean"]):.4f}
std:  {float(result["adjusted_rand_index"]["std"]):.4f}

raw: {repr(result)}
"""
print(report)

In [ ]:
output_report = output_dir / 'report.md'
output_report.write_text(report)

In [ ]:
result